In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.util import img_as_uint

import os
import tifffile
import pickle

from skimage.transform import estimate_transform, warp
from tqdm import tqdm

# Calculate transformation

In [9]:
reference_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_registration\reference_points'
transform_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_registration\transformation'

In [10]:
cycles = os.listdir(reference_dir)
cycles.sort()
# cycles = cycles[:]

In [11]:
station_reference = pd.read_csv(os.path.join(reference_dir, cycles[0]), index_col=0)

In [12]:
for cycle in tqdm(cycles[1:]):
    moving_reference = pd.read_csv(os.path.join(reference_dir, cycle), index_col=0)
    tform = estimate_transform('affine',
                               np.stack((moving_reference['col'].tolist(), moving_reference['row'].tolist()), axis=1),
                               np.stack((station_reference['col'].tolist(), station_reference['row'].tolist()),axis=1))
    with open(os.path.join(transform_dir, cycle.split('.')[0] + '.pkl'), 'wb') as f:
        pickle.dump(tform, f)

100%|██████████| 9/9 [00:00<00:00, 28.71it/s]


## Apply transformation

In [2]:
from joblib import Parallel, delayed

In [15]:
raw_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_registration\raw'
registered_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_registration\registered'

In [16]:
channels = ['C1','C2','C3','C4']
for c in channels:
    os.makedirs(os.path.join(registered_dir, c), exist_ok=True)

In [17]:
def transform_image(ch, cycle, shape, im_l):
    cy = cycle.split('.')[0]
    tform = pd.read_pickle(os.path.join(transform_dir, cy + '.pkl'))
    file_name = next(filter(lambda file: '_'+cy+'_' in file, im_l), None)
    img = tifffile.imread(os.path.join(raw_dir, ch, file_name))
    warped_image = warp(img, tform.inverse, output_shape=shape)
    tifffile.imsave(os.path.join(registered_dir, ch, file_name), img_as_uint(warped_image))

In [18]:
for ch in channels:
    im_l = os.listdir(os.path.join(raw_dir, ch))
    fix_file = next(filter(lambda file: '_'+'01'+'_' in file, im_l), None)
    fixed_img = tifffile.imread(os.path.join(raw_dir, ch, fix_file))
    _ = Parallel(n_jobs=4, backend='threading',verbose=1)(delayed(transform_image)(ch, cycle, fixed_img.shape, im_l) for cycle in cycles[1:])

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   9 out of   9 | elapsed:  2.6min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   9 out of   9 | elapsed:  2.6min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   9 out of   9 | elapsed:  2.6min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   9 out of   9 | elapsed:  2.4min finished


# Dot detection

In [2]:
from skimage.filters import threshold_triangle
from skimage.feature import peak_local_max
from skimage.util import img_as_uint, img_as_ubyte, img_as_float
import matplotlib.pyplot as plt

In [3]:
def detect_dots(img):
    thre_abs = threshold_triangle(img)*1.5
    return peak_local_max(img, min_distance=3,threshold_abs = thre_abs)

def filter_dots(dots, background):
    thre_background = threshold_triangle(background) * 2.5
    return dots[background[dots[:, 0], dots[:, 1]] < thre_background]

In [4]:
registered_im_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_registration\registered'
channels = ['C1', 'C2', 'C3', 'C4']
cycles = ['01','02','03','04','05','06','07','08','09','10','11','12','13','14','15']

In [5]:
ch = 'C1'
channel_dir = os.path.join(registered_im_dir, ch)
channel_fn_dict = {}
for im in os.listdir(channel_dir):
    if im.endswith('.tif'):
        cycle = im.split('_')[1]
        channel_fn_dict[cycle] = im[3:]

In [6]:
channel_fn_dict

{'01': '01_msg_84-1_ssa+_lyz_igha1_ighg2.tif',
 '02': '02_msg_84-1_ssa+_cxcl14_lum_col15a1.tif',
 '03': '03_msg_84-1_ssa+_rps26_rps20_rps10.tif',
 '04': '04_msg_84-1_ssa+_igkc_ighg1_ppia.tif',
 '05': '05_msg_84-1_ssa+_rpl39_rpl15_rps21.tif',
 '06': '06_msg_84-1_ssa+_sparc_cxcr3_col3a1.tif',
 '07': '07_msg_84-1_ssa+_cxcl11_tnfsf14_gapdh.tif',
 '08': '08_msg_84-1_ssa+_ido2_tgfb1_cxcl10.tif',
 '09': '09_msg_84-1_ssa+_tnfaip3_ifi44_birc3.tif',
 '10': '10_msg_84-1_ssa+_adra2a_rgs16_ubd.tif',
 '11': '11_msg_84-1_ssa+_ccr7_mki67_ccl19.tif',
 '12': '12_msg_84-1_ssa+_irf8_col1a1_hapln3.tif',
 '13': '13_msg_84-1_ssa+_mzb1_cxcl13_cxcl12.tif',
 '14': '14_msg_84-1_ssa+_ighm_tnfrsf4_cxcr5.tif',
 '15': '15_msg_84-1_ssa+_aqp5_empty_eef2.tif'}

In [7]:
genes = [['LYZ','IGHA1','IGHG2'],
         ['CXCL14','LUM','COL15A1'],
         ['RPS26','RPS20','RPS10'],
         ['IGKC','IGHG1','PPIA'],
         ['RPL39','RPL15','RPS21'],
         ['SPARC','CXCR3','COL3A1'],
         ['CXCL11','TNFSF14','GAPDH'],
         ['IDO2','TGFB1','CXCL10'],
         ['TNFAIP3','IFI44','BIRC3'],
         ['ADRA2A','RGS16','UBD'],
         ['CCR7', 'MKI67', 'CCL19'],
         ['IRF8', 'COL1A1', 'HAPLN3'],
         ['MZB1', 'CXCL13', 'CXCL12'],
         ['IGHM', 'EMPTY', 'CXCR5'],
         ['AQP5', 'EMPTY', 'EEF2']]
# 'TNFRSF4'

In [9]:
detected_dots = []
for i, cycle in tqdm(enumerate(cycles)):

    ch2 = tifffile.imread(os.path.join(registered_im_dir, 'C2', 'C2_'+channel_fn_dict[cycle]))
    ch2 = img_as_float(ch2)
    ch3 = tifffile.imread(os.path.join(registered_im_dir, 'C3', 'C3_'+channel_fn_dict[cycle]))
    ch3 = img_as_float(ch3)
    ch4 = tifffile.imread(os.path.join(registered_im_dir, 'C4', 'C4_'+channel_fn_dict[cycle]))
    ch4 = img_as_float(ch4)
    print('Images loaded')

    gene = genes[i]
    if not gene[0] == 'EMPTY':
        print('C2 detecting '+gene[0])
        ch2_dots = detect_dots(ch2)
        ch2_dot_filter = filter_dots(ch2_dots, ch3)
        print('C2 detected image')
    else:
        ch2_dots = np.zeros((2,2), dtype=int)
        print('C2 skipped')
    if not gene[1] == 'EMPTY':
        print('C3 detecting '+gene[1])
        ch3_dots = detect_dots(ch3)
        ch3_dot_filter = filter_dots(ch3_dots, ch2)
        print('C3 detected image')
    else:
        ch3_dots = np.zeros((2,2), dtype=int)
        print('C3 skipped')
    if not gene[2] == 'EMPTY':
        print('C4 detecting '+gene[2])
        ch4_dots = detect_dots(ch4)
        ch4_dot_filter = filter_dots(ch4_dots, ch3)
        print('C4 detected image')
    else:
        ch4_dots = np.zeros((2,2), dtype=int)
        print('C4 skipped')

    ch2_df = pd.DataFrame(ch2_dot_filter, columns=['row', 'col'])
    ch3_df = pd.DataFrame(ch3_dot_filter, columns=['row', 'col'])
    ch4_df = pd.DataFrame(ch4_dot_filter, columns=['row', 'col'])
    detected_dots.append([ch2_df, ch3_df, ch4_df])

0it [00:00, ?it/s]

Images loaded
C2 detecting LYZ
C2 detected image
C3 detecting IGHA1
C3 detected image
C4 detecting IGHG2


1it [01:00, 60.11s/it]

C4 detected image
Images loaded
C2 detecting CXCL14
C2 detected image
C3 detecting LUM
C3 detected image
C4 detecting COL15A1


2it [02:38, 82.51s/it]

...

C4 detected image
Images loaded
C2 detecting IGHM
C2 detected image
C3 skipped
C4 detecting CXCR5


14it [15:52, 55.75s/it]

C4 detected image
Images loaded
C2 detecting AQP5
C2 detected image
C3 skipped
C4 detecting EEF2


15it [16:30, 66.00s/it]

C4 detected image


In [10]:
n_cycles = 15
n_genes = 3
for cycle in detected_dots:
    for i in range(n_cycles):
        for j in range(n_genes):
            detected_dots[i][j]['gene'] = genes[i][j]

In [11]:
tnfrsf4_im = tifffile.imread(r"Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_registration\registered\C3\C3_14_msg_84-1_ssa+_ighm_tnfrsf4_cxcr5.tif")
# tnfrsf4_im = img_as_float(tnfrsf4_im)
thre = threshold_triangle(tnfrsf4_im)*2.5
# tnfrsf4_dots = peak_local_max(tnfrsf4_im, min_distance=3, threshold_abs=thre)

In [12]:
tnfrsf4_dots = peak_local_max(tnfrsf4_im, min_distance=30, threshold_abs=thre)

In [13]:
tnfrsf4_df = pd.DataFrame(tnfrsf4_dots, columns=['row', 'col'])

In [14]:
detected_dots[13][1] = tnfrsf4_df

In [15]:
detected_dots_df = (detected_dots[0] + detected_dots[1]
                    + detected_dots[2] + detected_dots[3]
                    + detected_dots[4] + detected_dots[5]
                    + detected_dots[6] + detected_dots[7]
                    + detected_dots[8] + detected_dots[9]
                    + detected_dots[10] + detected_dots[11]
                    + detected_dots[12] + detected_dots[13]
                    + detected_dots[14])
detected_dots_df = pd.concat(detected_dots_df, ignore_index=True)

In [16]:
detected_dots_df.to_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_analysis\detected_dots_whole_image.csv')

In [ ]:
# detected_dots_df = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_analysis\detected_dots_whole_image.csv', index_col=0)

# registration of additional cycles

## calculate transformation

In [ ]:
reference_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA-_174-1\00_registration\reference_points'
transform_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA-_174-1\00_registration\transformation'

In [38]:
cycles = os.listdir(reference_dir)
cycles.sort()
# cycles = cycles[:]

In [45]:
station_reference = pd.read_csv(os.path.join(reference_dir, cycles[0]), index_col=0)
# eliminate 4th row by index from station_reference
station_reference.drop(index=3, inplace=True)

In [47]:
cycles_to_register = cycles[10:15]

In [48]:
cycles_to_register

['11.csv', '12.csv', '13.csv', '14.csv', '15.csv']

In [49]:
for cycle in tqdm(cycles_to_register):
    moving_reference = pd.read_csv(os.path.join(reference_dir, cycle), index_col=0)
    moving_reference.drop(index=3, inplace=True)
    tform = estimate_transform('affine',
                               np.stack((moving_reference['col'].tolist(), moving_reference['row'].tolist()), axis=1),
                               np.stack((station_reference['col'].tolist(), station_reference['row'].tolist()),axis=1))
    with open(os.path.join(transform_dir, cycle.split('.')[0] + '.pkl'), 'wb') as f:
        pickle.dump(tform, f)

100%|██████████| 5/5 [00:00<00:00, 56.30it/s]


## apply transformation

In [50]:
from joblib import Parallel, delayed

In [65]:
raw_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA-_7202-1\00_registration\raw'
registered_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA-_7202-1\00_registration\registered'
transform_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA-_7202-1\00_registration\transformation'

In [66]:
channels = ['C1','C2','C3','C4']
for c in channels:
    os.makedirs(os.path.join(registered_dir, c), exist_ok=True)

In [67]:
def transform_image(ch, cycle, shape, im_l):
    cy = cycle.split('.')[0]
    tform = pd.read_pickle(os.path.join(transform_dir, cy + '.pkl'))
    file_name = next(filter(lambda file: '_'+cy+'_' in file, im_l), None)
    img = tifffile.imread(os.path.join(raw_dir, ch, file_name))
    warped_image = warp(img, tform.inverse, output_shape=shape)
    tifffile.imwrite(os.path.join(registered_dir, ch, file_name), img_as_uint(warped_image))

In [68]:
# cycles_to_register = ['12.csv']

In [69]:
for ch in channels:
    im_l = os.listdir(os.path.join(raw_dir, ch))
    fix_file = next(filter(lambda file: '_'+'01'+'_' in file, im_l), None)
    fixed_img = tifffile.imread(os.path.join(raw_dir, ch, fix_file))
    _ = Parallel(n_jobs=4, backend='threading',verbose=1)(delayed(transform_image)(ch, cycle, fixed_img.shape, im_l) for cycle in cycles_to_register)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 out of   5 | elapsed:   15.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 out of   5 | elapsed:   14.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 out of   5 | elapsed:   14.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 out of   5 | elapsed:   15.5s finished


## detect dots

In [17]:
registered_im_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_83-1\00_registration\registered'
channels = ['C1', 'C2', 'C3', 'C4']
cycles = ['11', '12', '13', '14', '15']

In [18]:
ch = 'C1'
channel_dir = os.path.join(registered_im_dir, ch)
channel_fn_dict = {}
for im in os.listdir(channel_dir):
    if im.endswith('.tif'):
        cycle = im.split('_')[1]
        channel_fn_dict[cycle] = im[3:]

In [19]:
channel_fn_dict

{'01': '01_msg_83-1_ssa+_lyz_igha1_ighg2.tif',
 '02': '02_msg_83-1_ssa+_cxcl14_lum_col15a1.tif',
 '03': '03_msg_83-1_ssa+_rps26_rps20_rps10.tif',
 '04': '04_msg_83-1_ssa+_igkc_ighg1_ppia.tif',
 '05': '05_msg_83-1_ssa+_rpl39_rpl15_rps21.tif',
 '06': '06_msg_83-1_ssa+_sparc_cxcr3_col3a1.tif',
 '07': '07_msg_83-1_ssa+_cxcl11_tnfsf14_gapdh.tif',
 '08': '08_msg_83-1_ssa+_ido2_tgfb1_cxcl10.tif',
 '09': '09_msg_83-1_ssa+_tnfaip3_ifi44_birc3.tif',
 '10': '10_msg_83-1_ssa+_adra2a_rgs16_ubd.tif',
 '11': '11_msg_83-1_ssa+_cccr7_mki67_ccl19.tif',
 '12': '12_msg_83-1_ssa+_irf8_col1a1_hapln3.tif',
 '13': '13_msg_83-1_ssa+_mzb1_cxcl13_cxcl12.tif',
 '14': '14_msg_83-1_ssa+_ighm_tnfrsf4_cxcr5.tif',
 '15': '15_msg_83-1_ssa+_aqp5_empty_eef2.tif'}

In [20]:
genes = [['CCR7', 'MKI67', 'CCL19'],
         ['IRF8', 'COL1A1', 'HAPLN3'],
         ['MZB1', 'CXCL13', 'CXCL12'],
         ['EMPTY', 'TNFRSF4', 'CXCR5'],
         ['AQP5', 'EMPTY', 'EEF2']]
# 'IGHM'

In [21]:
detected_dots = []
for i,cycle in tqdm(enumerate(cycles)):

    ch2 = tifffile.imread(os.path.join(registered_im_dir, 'C2', 'C2_'+channel_fn_dict[cycle]))
    ch2 = img_as_float(ch2)
    ch2 = ch2 / np.max(ch2)
    ch3 = tifffile.imread(os.path.join(registered_im_dir, 'C3', 'C3_'+channel_fn_dict[cycle]))
    ch3 = img_as_float(ch3)
    ch3 = ch3 / np.max(ch3)
    ch4 = tifffile.imread(os.path.join(registered_im_dir, 'C4', 'C4_'+channel_fn_dict[cycle]))
    ch4 = img_as_float(ch4)
    ch4 = ch4 / np.max(ch4)
    print('Images loaded')

    gene = genes[i]
    if not gene[0] == 'EMPTY':
        ch2_dots = detect_dots(ch2)
        ch2_dot_filter = filter_dots(ch2_dots, ch3)
        print('C2 detected')
    else:
        ch2_dots = np.zeros((2,2), dtype=int)
    if not gene[1] == 'EMPTY':
        ch3_dots = detect_dots(ch3)
        ch3_dot_filter = filter_dots(ch3_dots, ch2)
        print('C3 detected')
    else:
        ch3_dots = np.zeros((2,2), dtype=int)
    if not gene[2] == 'EMPTY':
        ch4_dots = detect_dots(ch4)
        ch4_dot_filter = filter_dots(ch4_dots, ch3)
        print('C4 detected')
    else:
        ch4_dots = np.zeros((2,2), dtype=int)
    
    ch2_df = pd.DataFrame(ch2_dot_filter, columns=['row', 'col'])
    ch3_df = pd.DataFrame(ch3_dot_filter, columns=['row', 'col'])
    ch4_df = pd.DataFrame(ch4_dot_filter, columns=['row', 'col'])
    detected_dots.append([ch2_df, ch3_df, ch4_df])

0it [00:00, ?it/s]

Images loaded
C2 detected
C3 detected


1it [01:38, 98.87s/it]

C4 detected
Images loaded
C2 detected
C3 detected


2it [03:31, 106.79s/it]

...

C4 detected
Images loaded
C3 detected


4it [06:04, 84.71s/it] 

C4 detected
Images loaded
C2 detected


5it [07:22, 88.43s/it]

C4 detected


In [22]:
ch2.shape

(26277, 26138)

In [114]:
tnfrsf4_im = tifffile.imread(r"Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_registration\registered\C3\C3_14_msg_84-1_ssa+_ighm_tnfrsf4_cxcr5.tif")
# tnfrsf4_im = img_as_float(tnfrsf4_im)
thre = threshold_triangle(tnfrsf4_im)*2.5
# tnfrsf4_dots = peak_local_max(tnfrsf4_im, min_distance=3, threshold_abs=thre)

In [115]:
tnfrsf4_dots = peak_local_max(tnfrsf4_im, min_distance=30, threshold_abs=thre)

In [116]:
tnfrsf4_df = pd.DataFrame(tnfrsf4_dots, columns=['row', 'col'])

In [26]:
n_cycles = 5
n_genes = 3
for cycle in detected_dots:
    for i in range(n_cycles):
        for j in range(n_genes):
            detected_dots[i][j]['gene'] = genes[i][j]

In [27]:
detected_dots[3][0] = tnfrsf4_df

In [28]:
detected_dots_df = (detected_dots[0] + detected_dots[1]
                    + detected_dots[2] + detected_dots[3]
                    + detected_dots[4])
detected_dots_df = pd.concat(detected_dots_df, ignore_index=True)

In [29]:
existing_dots = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_83-1\00_analysis\detected_dots_whole_image.csv', index_col=0)

In [30]:
old_genes = [['LYZ','IGHA1','IGHG2'],
         ['CXCL14','LUM','COL15A1'],
         ['RPS26','RPS20','RPS10'],
         ['IGKC','IGHG1','PPIA'],
         ['RPL39','RPL15','RPS21'],
         ['SPARC','CXCR3','COL3A1'],
         ['CXCL11','TNFSF14','GAPDH'],
         ['IDO2','TGFB1','CXCL10'],
         ['TNFAIP3','IFI44','BIRC3'],
         ['ADRA2A','RGS16','UBD']]
old_genes = [item for sublist in old_genes for item in sublist]
existing_dots = existing_dots[existing_dots['gene'].isin(old_genes)]

In [31]:
detected_dots_df = detected_dots_df[detected_dots_df['gene'] != 'EMPTY']

In [32]:
detected_dots_df = pd.concat([existing_dots, detected_dots_df], ignore_index=True)

In [34]:
detected_dots_df.to_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_83-1\00_analysis\detected_dots_whole_image.csv')

In [117]:
detected_dots_df = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_analysis\detected_dots_whole_image.csv', index_col=0)

In [118]:
tnfrsf4_df['gene'] = 'TNFRSF4'

In [119]:
detected_dots_df = detected_dots_df[detected_dots_df['gene'] != 'EMPTY']
detected_dots_df = detected_dots_df[detected_dots_df['gene'] != 'TNFRSF4']
detected_dots_df = pd.concat([detected_dots_df, tnfrsf4_df], ignore_index=True)

In [120]:
detected_dots_df = detected_dots_df[detected_dots_df['gene'] == detected_dots_df['gene']]

In [121]:
detected_dots_df.to_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_analysis\detected_dots_whole_image.csv')

# counting

In [102]:
from skimage.measure import regionprops_table
from skimage import io
import pandas as pd
import tifffile

In [122]:
detected_dots_df = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_analysis\detected_dots_whole_image.csv', index_col=0)

In [123]:
mask = tifffile.imread(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_segmentation\C1_01_msg_84-1_ssa+_lyz_igha1_ighg2_cp_masks.tif')

In [124]:
# expand the mask
from scipy.ndimage import distance_transform_edt, label
from skimage.segmentation import expand_labels

In [125]:
expanded = expand_labels(mask, distance=5)

In [126]:
detected_dots_df['label'] = expanded[detected_dots_df['row'], detected_dots_df['col']]

In [ ]:
detected_dots_df_filtered = detected_dots_df[detected_dots_df['label'] > 0]

In [128]:
detected_dots_df.shape

(1262415, 4)

In [129]:
genes = detected_dots_df['gene']

In [130]:
detected_dots_df_filtered.to_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_analysis\detected_dots_filtered.csv')

In [131]:
detected_dots_df_filtered['gene'].unique().shape

(44,)

In [132]:
detected_dots_df_filtered['gene'].value_counts().shape

(44,)

In [133]:
genes = detected_dots_df['gene'].unique().tolist()
gene_count = {}
for gene in genes:
    gene_df = detected_dots_df_filtered[detected_dots_df_filtered['gene'] == gene]
    gene_count[gene] = gene_df['label'].value_counts()

In [134]:
gene_count_df = pd.DataFrame(gene_count)
gene_count_df = gene_count_df.fillna(0)

In [135]:
gene_count_df

,LYZ,IGHA1,IGHG2,CXCL14,LUM,COL15A1,RPS26,RPS20,RPS10,IGKC,...,COL1A1,HAPLN3,MZB1,CXCL13,CXCL12,IGHM,CXCR5,AQP5,EEF2,TNFRSF4
label,,,,,,,,,,,,,,,,,,,,,
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73787,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
73788,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
73789,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [136]:
# gene_count_df.drop(columns=[np.nan], inplace=True)

In [137]:
cells_properties = regionprops_table(mask, properties=['label', 'area', 'centroid'])
cells_properties_df = pd.DataFrame(cells_properties)

In [138]:
cells_properties_df.rename(columns={'centroid-0': 'row', 'centroid-1': 'col'}, inplace=True)

In [139]:
cells_properties_df.index = cells_properties_df['label']

In [140]:
# merge the two dataframes
cells_properties_df = cells_properties_df.merge(gene_count_df, left_index=True, right_index=True)
cells_properties_df = cells_properties_df.fillna(0)

In [141]:
cells_properties_df.to_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_analysis\expression.csv')

# stromal annotation

In [300]:
cells_properties_df = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_83-1\00_analysis\expression.csv')
# cells_properties_df_annotated = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_83-1\00_analysis\cells_properties_df_annotated.csv')

In [301]:
top_10_percent_threshold = cells_properties_df['LUM'].quantile(0.9)
cell_types = ['Stromal' if cells_properties_df.loc[cell, 'LUM'] > top_10_percent_threshold else 'Non-stromal' for cell in cells_properties_df.index]
cells_properties_df['cell_types'] = cell_types

In [302]:
cells_properties_df.to_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_83-1\00_analysis\cells_properties_df_annotated.csv')

In [211]:
cells_properties_df = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA-_174-1\00_analysis\expression.csv', index_col=0)
cells_properties_df_annotated = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA-_174-1\00_analysis\cells_properties_df_annotated.csv')

In [214]:
cells_properties_df['cell_types'] = cells_properties_df_annotated['cell_types']

In [215]:
cells_properties_df.to_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA-_174-1\00_analysis\cells_properties_df_annotated.csv')

In [240]:
cells_properties_df = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_analysis\expression.csv')
cells_properties_df_annotated = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA+_84-1\00_analysis\cells_properties_df_annotated.csv')

In [225]:
cells_properties_df = cells_properties_df[cells_properties_df['label'].isin(cells_properties_df_annotated['label'].tolist())]

In [226]:
cells_properties_df.shape

(60779, 51)

In [227]:
cells_properties_df_annotated.shape

(65862, 36)

In [241]:
cells_properties_df_annotated

,label,label.1,area,row,col,LYZ,IGHA1,IGHG2,CXCL14,LUM,...,IDO2,TGFB1,CXCL10,TNFAIP3,IFI44,BIRC3,ADRA2A,RGS16,UBD,cell_types
0,1,1,1033,963.207164,18451.999032,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Non-stromal
1,2,2,1343,977.313477,18479.977662,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Non-stromal
2,3,3,1888,977.950212,18358.344809,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Non-stromal
3,4,4,1495,974.753846,18409.632107,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Non-stromal
4,5,5,1474,984.126866,18530.097015,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Non-stromal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65857,73787,73787,807,15671.682776,11751.697646,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Non-stromal
65858,73788,73788,885,15670.092655,12124.745763,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,Non-stromal
65859,73789,73789,585,15677.104274,12084.755556,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Non-stromal
65860,73790,73790,1630,15715.346626,12108.753374,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Non-stromal


In [242]:
cells_properties_df

,label,label.1,area,row,col,LYZ,IGHA1,IGHG2,CXCL14,LUM,...,MZB1,CXCL13,CXCL12,IGHM,TNFRSF4,CXCR5,AQP5,EMPTY,EEF2,Unnamed: 50
0,10,10,1526.0,1057.753604,13378.716907,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,11,11,20.0,1043.650000,13293.300000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,12,12,1110.0,1125.927928,13433.175676,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,13,13,1386.0,1168.783550,16469.352092,20.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,14,14,260.0,1175.361538,13401.030769,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68003,73482,73482,1742.0,24636.782434,15890.440299,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
68004,73484,73484,1088.0,24665.954044,15834.240809,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
68005,73486,73486,1460.0,24747.442466,16050.199315,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
68006,73488,73488,345.0,24776.649275,16382.081159,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [271]:
cells_properties_df = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA-_7202-1\00_analysis\expression.csv')
cells_properties_df_annotated = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA-_7202-1\00_analysis\cells_properties_df_annotated.csv')

In [272]:
cells_properties_df.shape

(22267, 49)

In [273]:
cells_properties_df_annotated.shape

(22257, 36)

In [277]:
cells_properties_df.shape

(22257, 49)

In [278]:
cells_properties_df['cell_types'] = cells_properties_df_annotated['cell_types'].tolist()

In [279]:
cells_properties_df.to_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20240918_xenium_hcr\SSA-_7202-1\00_analysis\cells_properties_df_annotated.csv')